# Notebook 2: Model Training & Evaluation
**Answers RQ1:** How do Rule-Based, Logistic Regression, XGBoost, and Neural Network models differ in fraud detection performance?

## Steps
1. Load and preprocess data (SMOTE for imbalance)
2. Train all four models
3. Evaluate on held-out test set
4. Visualise comparative performance

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve

from utils.data_loader import DataLoader, LABEL_COLUMN
from utils.preprocessing import Preprocessor
from models.rule_engine import RuleBasedDetector
from training.train_models import train_logistic_regression, train_xgboost, train_neural_network, save_artefacts
from training.evaluate import Evaluator

sns.set_theme(style='whitegrid')
os.makedirs('../outputs', exist_ok=True)
%matplotlib inline

## 1. Load & Split Data

In [ ]:
DATA_PATH = '../data/creditcard.csv'
loader = DataLoader(data_path=DATA_PATH if os.path.exists(DATA_PATH) else None)
df = loader.load()
train_df, val_df, test_df = loader.split(df)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## 2. Preprocess (SMOTE)

In [ ]:
preprocessor = Preprocessor()
X_train, y_train = preprocessor.fit_transform(train_df, apply_smote=True)
X_val,   y_val   = preprocessor.transform(val_df)
X_test,  y_test  = preprocessor.transform(test_df)

print(f'X_train shape: {X_train.shape}, fraud: {y_train.sum():.0f}/{len(y_train)}')

## 3. Train Models

In [ ]:
lr_model  = train_logistic_regression(X_train, y_train)
xgb_model = train_xgboost(X_train, y_train)
nn_model  = train_neural_network(X_train, y_train, X_val, y_val)
print('Training complete')

## 4. Evaluate All Models

In [ ]:
evaluator = Evaluator()

# Rule-based: use preprocessed single features
rb = RuleBasedDetector()
rb_probs = np.array([rb.predict(row)[1] for row in test_df.to_dict('records')])
rb_metrics  = evaluator.evaluate_from_probs(rb_probs,  y_test, 'Rule-Based')
lr_metrics  = evaluator.evaluate(lr_model,  X_test, y_test, 'Logistic Regression')
xgb_metrics = evaluator.evaluate(xgb_model, X_test, y_test, 'XGBoost')

all_metrics = [rb_metrics, lr_metrics, xgb_metrics]

if nn_model is not None:
    nn_probs = nn_model.predict(X_test, verbose=0).ravel()
    nn_metrics = evaluator.evaluate_from_probs(nn_probs, y_test, 'Neural Network')
    all_metrics.append(nn_metrics)

for m in all_metrics:
    evaluator.print_report(m)

## 5. Comparative Bar Chart

In [ ]:
import pandas as pd

rows = [{
    'Model': m.model_name,
    'Precision': m.precision,
    'Recall': m.recall,
    'F1': m.f1_score,
    'ROC-AUC': m.roc_auc,
} for m in all_metrics]
comp_df = pd.DataFrame(rows).set_index('Model')

ax = comp_df.plot(kind='bar', figsize=(10, 5), width=0.75)
ax.set_title('Model Comparison – Test Set')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', dpi=150)
plt.show()

display(comp_df.round(4))

## 6. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

prob_map = {
    'Rule-Based': rb_probs,
    'Logistic Regression': lr_model.predict_proba(X_test)[:,1],
    'XGBoost': xgb_model.predict_proba(X_test)[:,1],
}
if nn_model is not None:
    prob_map['Neural Network'] = nn_model.predict(X_test, verbose=0).ravel()

for name, probs in prob_map.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr, label=name)

ax.plot([0,1],[0,1],'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/roc_curves.png', dpi=150)
plt.show()

## 7. Save Artefacts

In [ ]:
save_artefacts(lr_model, xgb_model, nn_model, preprocessor)
print('Models saved to ../models/saved/')